# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [5]:
import gradio as gr
import os
from openai import OpenAI
from dotenv import load_dotenv


load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
gpt_client = OpenAI(api_key=openai_api_key)
llama_client = OpenAI(base_url="http://localhost:11434/v1/" , api_key="llama3.2")


MODELS_CLIENTS = {
    "openai" : gpt_client ,
    "llama" : llama_client
}
MODELS_NAME= {
  "openai" : "gpt-4.1-mini" , 
  "llama" : "llama3.2"
 }


system_prompt  = "you are a helpfull assistant and your jop is to explain the code that user gives to you step by step"


def chat (user_prompt , history , client_choice):
    client_selection  = MODELS_CLIENTS[client_choice]
    model = MODELS_NAME[client_choice]
    chat_history = [{"role" : h["role"] , "content" : h["content"]} for h in history]
    messages = [{"role" : "system" , "content" : system_prompt}] +chat_history + [{"role" :"user" , "content" :user_prompt}]
    answer = client_selection.chat.completions.create(model=model , messages=messages , stream=True)
    response = ''
    for chunk in answer:
        response+= chunk.choices[0].delta.content or ''
        yield response
gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[gr.Dropdown(choices=list(MODELS_CLIENTS.keys()), value="openai", label="MODEL")],
    additional_inputs_accordion=gr.Accordion(label="Settings", open=True),
).launch(inbrowser=True)      
    




* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
